# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [25]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [26]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../02_activities/documents/ai_report_2025.pdf")
documents = loader.load()

In [27]:
document_text = ""
for page in documents:
    document_text += page.page_content + "\n"

# print(document_text)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [32]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os

# Configuration
MODEL = "gpt-4o-mini"
GATEWAY_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"
API_KEY = os.getenv('API_GATEWAY_KEY')

# Pydantic model
class DocumentAnalysis(BaseModel):
    author: str = Field(description="Author of the document")
    title: str = Field(description="Title of the document")
    relevance: str = Field(description="Why this is relevant for AI professionals")
    summary: str = Field(description="Summary in cricket commentator tone, max 1000 tokens")
    tone: str = Field(description="The tone used in the summary")
    input_tokens: int = Field(description="Number of input tokens")
    output_tokens: int = Field(description="Number of output tokens")

# System prompt (instructions)
SYSTEM_PROMPT = """You are an expert document analyst. Analyze the provided document and extract structured insights.
Your response must be valid JSON matching this exact structure:
{
    "author": "string",
    "title": "string",
    "relevance": "string (one paragraph max)",
    "summary": "string (max 1000 tokens)",
    "tone": "Cricket Commentator"
}"""

# User prompt template
USER_PROMPT_TEMPLATE = """Please analyze this document and write the summary as if you were a cricket commentator narrating a first class test match.

Document Content:
{document_text}"""

def get_client() -> OpenAI:
    """Initialize OpenAI client with gateway."""
    return OpenAI(
        base_url=GATEWAY_URL,
        api_key="any-value",
        default_headers={"x-api-key": API_KEY}
    )

def analyze_document(client: OpenAI, document_text: str) -> DocumentAnalysis:
    """Analyze document with structured output."""
    
    # Format user prompt with document
    user_prompt = USER_PROMPT_TEMPLATE.format(document_text=document_text)
    
    # Call OpenAI with structured output
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        response_format=DocumentAnalysis
    )
    
    # Extract analysis and add token counts
    analysis = response.choices[0].message.parsed
    analysis.input_tokens = response.usage.prompt_tokens
    analysis.output_tokens = response.usage.completion_tokens
    
    return analysis

# Usage
client = get_client()
result = analyze_document(client, document_text)
print(result.model_dump_json(indent=2))

{
  "author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "This comprehensive report is crucial for AI professionals as it highlights the current landscape of AI implementation across industries. Understanding the GenAI Divide can guide organizations looking to maximize their AI investments and navigate the complexities of deployment effectively.",
  "summary": "Ladies and gentlemen, welcome to the riveting match of the GenAI Divide! We've seen tremendous enthusiasm from enterprises, with a staggering $30 to $40 billion poured into generative AI. Yet, as we dive into the statistics, we find that a mere 5% of organizations are truly reaping the rewards of their investments. Yes, you heard me right – 95% of pilots are having a hard time scoring any runs on the P&L scoreboard, and many players find themselves caught in a tight spot between exploration and execution. But don’t be f

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
